# Exterior electrostatics

A conducting disk of radius $1/4$ centered at $(0,1)$ has potential one. The line $y=0$ is grounded. The surrounding upper half-plane is unbounded, so we solve on

$$
\begin{align*}
\Omega_R=\big((-R,R)\times(0,R)\big)\setminus\overline{B_{1/4}((0,1))}.
\end{align*}
$$

The potential satisfies $-\Delta u=0$. On the three artificial outer edges we compare $u=0$ (**Dirichlet**) and $\partial_nu=0$ (**Neumann**). The physical data on the conductor and ground are identical in both cases.

We compute the energy $E=\int_{\Omega_R}|\nabla u|^2\,dx$.

## Packages

NGSolve/Netgen and Matplotlib are needed.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ngsolve import *
from netgen.occ import WorkPlane, OCCGeometry, Y

## Geometry

`ground` labels the bottom edge, `outer` the other three rectangle edges, and `conductor` the circular hole. The hole is removed from the computational domain.

`maxh` controls the global mesh size, while `conductor_h` resolves the circle locally. Keep both fixed while changing $R$, so that moving the outer boundary does not coarsen the mesh near the conductor. The mesh geometry has order three.

In [ ]:
def exterior_mesh(R, maxh=0.4, conductor_h=0.04):
    """Rectangle above the ground, with a circular conductor removed."""
    outer = WorkPlane().MoveTo(-R, 0).Rectangle(2*R, R).Face()
    outer.edges.name = "outer"
    outer.edges.Min(Y).name = "ground"
    conductor = WorkPlane().MoveTo(0, 1).Circle(0.25).Face()
    conductor.edges.name = "conductor"
    conductor.edges.maxh = conductor_h
    geometry = OCCGeometry(outer-conductor, dim=2)
    mesh = Mesh(geometry.GenerateMesh(maxh=maxh))
    mesh.Curve(3)
    return mesh

## Solver

Both problems minimize $\int|\nabla v|^2$ with the prescribed Dirichlet values. The weak equation is

$$
\begin{align*}
\int_{\Omega_R}\nabla u\cdot\nabla v\,dx=0
\end{align*}
$$

for test functions vanishing on the Dirichlet boundary. Leaving `outer` out of the Dirichlet selection gives the homogeneous Neumann condition there.

The helper uses cubic finite elements. It first sets the conductor value and then corrects the free degrees of freedom. You can use this helper without changing its implementation.

In [ ]:
def solve_potential(mesh, outer_bc, order=3):
    """Solve Laplace's equation with conductor=1 and ground=0."""
    if outer_bc not in ("Dirichlet", "Neumann"):
        raise ValueError("outer_bc must be Dirichlet or Neumann")
    essential = "ground|conductor"
    if outer_bc == "Dirichlet":
        essential += "|outer"
    fes = H1(mesh, order=order, dirichlet=essential)
    u, v = fes.TnT()
    a = BilinearForm(fes, symmetric=True)
    a += grad(u)*grad(v)*dx
    a.Assemble()
    uh = GridFunction(fes)
    uh.Set(CF(1), definedon=mesh.Boundaries("conductor"))
    rhs = uh.vec.CreateVector()
    rhs.data = -a.mat*uh.vec
    uh.vec.data += a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")*rhs
    return uh

## 1. Move the artificial boundary

For $R=2,3,4,6$, generate one mesh and solve both boundary cases on it. Compute each energy using

```python
Integrate(grad(uh)*grad(uh), mesh)
```

Store the results and print a table. Plot both energies against $R$ and describe how the choice of artificial boundary condition affects the energies and their gap.

Optional supplied reference: the exact exterior energy is $2\pi/\operatorname{arcosh}(4)$. You do not need to derive this formula.

In [ ]:
radii = [2, 3, 4, 6]
truncation_results = []
# TODO: Generate a mesh for each R and solve both boundary cases.
# TODO: Compute, store, and print the two energies and their gap.

In [ ]:
# TODO: Plot both energies against R. Add the optional exterior reference.

## 2. Check that the truncation effect is resolved

At $R=4$, repeat both solves with `maxh=0.2` and `conductor_h=0.02`. Compare the energy changes with the gap between the Dirichlet and Neumann energies on the original mesh.

In [ ]:
# TODO: Repeat both R=4 computations with both mesh sizes halved.
# TODO: Compare the energy changes with the truncation gap.